Before 2017, if you wanted to translate a sentence from English to German, the standard approach was an encoder-decoder Recurrent Neural Network (RNN), including its more advanced variants, Long Short-Term Memory (LSTM) and Gated Recurrent Units (GRU) (Cho et al., 2014; Bahdanau et al., 2014). Neural machine translation is a sequence-to-sequence prediction task, where the encoder reads the sequence of source tokens, and the decoder is expected to predict each target token based on what the encoder has read and on the target tokens it has already generated.

<div align="center">
<img src="images/seq2seq.png" width="600" style="display:block; margin:0 auto;">
<p style="margin-top:4px;"><em>Fig 1: The unrolled encoder-decoder architecture for neural machine translation from English to German.</em></p>
</div>

- The encoder is a nonlinear function $f_{enc}$, e.g. an RNN, that reads the input sentence, represented as a sequence of tokens, one token at a time. 

  Let $\mathbf{x} = (x_1, \ldots, x_{T_x})$ be a sequence of tokens, where $x_i \in \mathbb{R}^{K_x}$ and $K_x$ is the source vocabulary size.

  At each step $t$, it produces a hidden state $h_t$ - a fixed-size vector summarizing everything read so far, up to and including $x_t$ - as shown in the encoding sequence below (Bahdanau et al., 2014, §2.1).

  $$x_1 \xrightarrow{h_1} x_2 \xrightarrow{h_2} x_3 \xrightarrow{h_3} \cdots \xrightarrow{h_{T_x-1}} x_{T_x} \xrightarrow{h_{T_x}}$$

  The encoder's hidden state $h_t$ is computed as a function of the current token $x_t$ and the previous hidden state $h_{t-1}$. It is represented as:

  $$h_t = f_{enc}(x_t, h_{t-1}), \quad f_{enc}: \mathbb{R}^{K_x} \times \mathbb{R}^n \to \mathbb{R}^n \text{ nonlinear and differentiable}$$

  where $n$ is the number of hidden units. $f_{enc}$ maps back into the same space $h_{t-1}$ occupies, which is what allows the recurrence to be applied repeatedly across every time step; differentiability is required so gradients can be backpropagated through the unrolled recurrence during training.

  
- After reading the entire sequence, the encoder's final hidden state $h_{T_x}$ is taken as a fixed-length numerical vector, called the context vector $c$ (Bahdanau et al., 2014, Eq. 1; Sutskever et al., 2014).

  $$c = q(\{h_1, \ldots, h_{T_x}\}) = h_{T_x}, \quad q_{T_x}: (\mathbb{R}^n)^{T_x} \to \mathbb{R}^n \text{ nonlinear and differentiable}$$

  Because $T_x$ varies with the length of the input sentence, $q$ is not a single fixed-arity function but a family $q = \{q_{T_x}\}_{T_x \geq 1}$, one map per possible sequence length, each reducing its (fixed-length) tuple of annotations to a single vector in $\mathbb{R}^n$. Equivalently, one can define $q$ on the union of finite-length tuples, $q: \bigcup_{k \geq 1} (\mathbb{R}^n)^k \to \mathbb{R}^n$, so that a single function correctly accepts input of any length. Either way, $n$ matches the encoder's hidden state size, and $q$ takes the *entire set* of annotations $\{h_1,\ldots,h_{T_x}\}$ - of variable size, depending on sentence length - and reduces it to one fixed-size vector.

- The decoder is also a nonlinear function $f_{dec}$, e.g. an RNN, that predicts the next output token $y_t$ (the $t$-th word of the translation) based on the latest generated token $y_{t-1}$, its previous hidden state $s_{t-1}$, and the context vector $c$, as shown in the decoding sequence below. This is the fixed-context formulation that Bahdanau et al. (2014, §2.1) present as the standard RNN encoder-decoder setup *before* introducing their attention mechanism in §3, where the single vector $c$ is replaced by a per-step context $c_t$ that varies with the decoder's position (Bahdanau et al., 2014, §2.1, Eqs. 2-3).

  Let $\mathbf{y} = (y_1, \ldots, y_{T_y})$ be a sequence of output tokens, where $y_i \in \mathbb{R}^{K_y}$ and $K_y$ is the target vocabulary size.

  $$
  \begin{array}{ccccccccc}
  y_1 & & y_2 & & y_3 & & y_4 & & \\
  \big\uparrow & \searrow & \big\uparrow & \searrow & \big\uparrow & \searrow & \big\uparrow & \searrow & \\
  s_1 & \xrightarrow{} & s_2 & \xrightarrow{} & s_3 & \xrightarrow{} & s_4 & \xrightarrow{} & \cdots \\
  \big\uparrow & & \big\uparrow & & \big\uparrow & & \big\uparrow & & \\
  c & & c & & c & & c & &
  \end{array}
  $$

  At each step $t$, the decoder's hidden state $s_t$ is updated according to:

  $$s_t = f_{dec}(s_{t-1}, y_{t-1}, c), \quad f_{dec}: \mathbb{R}^n \times \mathbb{R}^{K_y} \times \mathbb{R}^n \to \mathbb{R}^n \text{ nonlinear and differentiable}$$

  matching the same hidden size $n$ used by the encoder, so that $c$ and $s_t$ are compatible when combined.

  The decoder does not predict $y_t$ directly from $s_t$ alone - instead, it defines a probability distribution over the translation $\mathbf{y}$ by decomposing the joint probability into ordered conditionals (Bahdanau et al., 2014, Eq. 2):

  $$p(\mathbf{y}) = \prod_{t=1}^{T_y} p(y_t \mid \{y_1, \ldots, y_{t-1}\}, c)$$

  $s_t$ is just a memory vector - it isn't itself a probability distribution, so it can't directly tell us which word comes next. We need one more step to turn that memory into an actual guess over the vocabulary. That's the job of a function $g$: it takes $s_t$ (along with $y_{t-1}$ and $c$) and outputs a probability for every possible word, from which $y_t$ is picked (Bahdanau et al., 2014, Eq. 3):

  $$p(y_t \mid \{y_1, \ldots, y_{t-1}\}, c) = g(y_{t-1}, s_t, c)$$

To translate a paragraph correctly, the model must understand context and grammatical dependencies between words - even when those words are far apart in the sentence. But the core mechanism of RNNs is inherently sequential: the hidden state $h_t$ can only be computed once $h_{t-1}$ is known, which itself required $h_{t-2}$, and so on.

This creates two problems:

**1. Sequential computation prevents parallelization**

Because the hidden state at the current time step $h_t$ depends on the previous hidden state $h_{t-1}$, positions within a sequence cannot be computed simultaneously (Vaswani et al., 2017). This prevents parallelization within training examples, which becomes especially limiting when training on large amounts of data under constrained computation or memory resources.

**2. Long-range dependencies are difficult to learn**

Standard RNNs, and even their gated variants - LSTM (Hochreiter & Schmidhuber, 1997) and GRU (Cho et al., 2014) - compress all preceding context into a single fixed-size hidden state. Information relevant at position $t$ must survive being repeatedly overwritten across every intermediate step to influence a distant position $t+k$, and ultimately to survive all the way into the context vector $c$. If information about an early token is not retained in $c$, the decoder has no way to recover it, resulting in a loss of information in the output sequence. Hochreiter et al. (2001) showed that this leads to vanishing/exploding gradients, making it difficult for RNNs to learn dependencies between positions that are far apart in the sequence. If some of the information in the compressed encoded vector is missing, how much can we rely on each token at the source sequence to predict a particular output token remains in question.

Bahdanau et al. (2014) addressed the second problem by introducing an attention mechanism: at each decoding step, the decoder could attend over all encoder hidden states — here called annotations, since each one is retained and made individually available for the decoder to consult, rather than only contributing to one final compressed vector $c$.

Instead of using one fixed $c$ at every step, the decoder computes a fresh context vector $c_i$ at every decoding step $i$, as a weighted sum of these annotations (Bahdanau et al., 2014, Eq. 5):

$$c_i = \sum_{j=1}^{T_x} \alpha_{ij} h_j$$

The weight $\alpha_{ij}$ assigned to each annotation $h_j$ is obtained by normalizing a set of scores with a softmax, so that the weights are non-negative and sum to 1 (Bahdanau et al., 2014, Eq. 6):

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})}$$

Since the weights $\alpha_{ij}$ are non-negative and sum to 1 across $j$, the sum $c_i = \sum_j\alpha_{ij}h_j$ can be understood as an expected annotation, where the expectation is taken over possible alignments - $\alpha_{ij}$ playing the role of the probability that target word $y_i$ is aligned to (translated from) source word $x_j$ (Bahdanau et al., 2014, §3.1). Rather than committing to one single source word as the correct alignment, the model blends information from all source positions, weighted by how relevant each one currently is.

Each raw score $e_{ij}$ is computed by an alignment model $a$, comparing the decoder's previous hidden state $s_{i-1}$ against a single encoder annotation $h_j$ (Bahdanau et al., 2014, §3.1):

$$e_{ij} = a(s_{i-1}, h_j)= v_a^\top \tanh(W_a s_{i-1} + U_a h_j)$$

where $W_a$, $U_a$, and $v_a$ are learned weight matrices and vector, jointly trained end to end with the rest of the model. It is not hand designed, and its behavior as a compatibility score between source and target positions emerges purely from training on translation data (Bahdanau et al., 2014, §3.1).

The attention score $e_{ij}$ indicates how much we can rely on the each source position $x_j$ to predict a particular target prediction $y_i$. 

$a$ is parametrized as a small feedforward neural network, jointly trained end-to-end with the rest of the model - it is not hand-designed, and its behavior as a "compatibility score" between source and target positions emerges purely from training on translation data (Bahdanau et al., 2014, §3.1).


**The Transformer's proposal**

Vaswani et al. (2017) proposed removing recurrence entirely and relying solely on attention: "In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output". This directly resolves both problems:

- *Parallelization*: since no position's representation depends on first computing another position's, all positions can be processed simultaneously.
- *Path length*: self-attention connects any two positions with O(1) sequential operations, compared to O(n) for a recurrent layer (Vaswani et al., 2017, Table 1), making long-range dependencies no harder to learn than short-range ones.


**References**

- Bahdanau, D., Cho, K., & Bengio, Y. (2014). *Neural Machine Translation by Jointly Learning to Align and Translate*. arXiv:1409.0473.
- Cho, K., et al. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. arXiv:1406.1078.
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.
- Hochreiter, S., Bengio, Y., Frasconi, P., & Schmidhuber, J. (2001). *Gradient Flow in Recurrent Nets: The Difficulty of Learning Long-Term Dependencies*.
- Sutskever, I., Vinyals, O., & Le, Q. V. (2014). *Sequence to Sequence Learning with Neural Networks*. In *Advances in NeurIPS 27*.
- Vaswani, A., et al. (2017). *Attention Is All You Need*. In *Advances in NeurIPS 30*.